In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-06-28 16:03:43,929] A new study created in memory with name: no-name-f2580863-8e15-45d4-aa3a-81b3d488b067
[I 2026-06-28 16:03:44,267] Trial 0 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 141, 'max_depth': 6}. Best is trial 0 with value: 0.7597765363128491.
[I 2026-06-28 16:03:44,726] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 196, 'max_depth': 7}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-06-28 16:03:45,006] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 114, 'max_depth': 12}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-06-28 16:03:45,468] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 197, 'max_depth': 13}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-06-28 16:03:45,605] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 58, 'max_depth': 7}. Best is trial 2 with value: 0.77467411

In [7]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7839851024208566
Best hyperparameters: {'n_estimators': 73, 'max_depth': 18}


In [8]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## Samplers in Optuna

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-06-28 16:04:09,377] A new study created in memory with name: no-name-1525dc19-c059-4ade-bf6b-15ad6f165b74
[I 2026-06-28 16:04:09,704] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 16:04:10,139] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 181, 'max_depth': 7}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-06-28 16:04:10,530] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 177, 'max_depth': 4}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-06-28 16:04:10,995] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 197, 'max_depth': 11}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-06-28 16:04:11,283] Trial 4 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 118, 'max_depth': 17}. Best is trial 4 with value: 0.7783985

In [11]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 119, 'max_depth': 7}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)


# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
    
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-06-28 16:04:30,847] A new study created in memory with name: no-name-88e1dd69-d706-40e2-98e9-566892c04926
[I 2026-06-28 16:04:31,082] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 16:04:31,453] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 16:04:31,583] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-28 16:04:31,836] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-28 16:04:32,081] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# 3. Slice Plot
plot_slice(study).show()

In [21]:
# 4. Contour Plot
plot_contour(study).show()

In [22]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [25]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-28 16:05:10,731] A new study created in memory with name: no-name-9158f5e4-55e2-4952-aa62-459db86785f4
[I 2026-06-28 16:05:10,749] Trial 0 finished with value: 0.7281191806331471 and parameters: {'classifier': 'SVM', 'C': 1.278120209033799, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7281191806331471.
[I 2026-06-28 16:05:11,376] Trial 1 finished with value: 0.7690875232774674 and parameters: {'classifier': 'RandomForest', 'n_estimators': 269, 'max_depth': 15, 'min_samples_split': 7, 'min_samples_leaf': 5, 'bootstrap': True}. Best is trial 1 with value: 0.7690875232774674.
[I 2026-06-28 16:05:11,565] Trial 2 finished with value: 0.7821229050279329 and parameters: {'classifier': 'RandomForest', 'n_estimators': 97, 'max_depth': 6, 'min_samples_split': 9, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 2 with value: 0.7821229050279329.
[I 2026-06-28 16:05:11,979] Trial 3 finished with value: 0.7541899441340782 and parameters: {'classifier': 'Grad

In [26]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.13777175450640633, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.728119,2026-06-28 16:05:10.733239,2026-06-28 16:05:10.749367,0 days 00:00:00.016128,1.278120,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.769088,2026-06-28 16:05:10.750360,2026-06-28 16:05:11.375990,0 days 00:00:00.625630,NaN,True,RandomForest,NaN,NaN,NaN,15.0,5.0,7.0,269.0,COMPLETE
2,2,0.782123,2026-06-28 16:05:11.376640,2026-06-28 16:05:11.565717,0 days 00:00:00.189077,NaN,False,RandomForest,NaN,NaN,NaN,6.0,6.0,9.0,97.0,COMPLETE
3,3,0.754190,2026-06-28 16:05:11.566356,2026-06-28 16:05:11.979481,0 days 00:00:00.413125,NaN,NaN,GradientBoosting,NaN,NaN,0.161676,4.0,6.0,8.0,178.0,COMPLETE
4,4,0.728119,2026-06-28 16:05:11.980227,2026-06-28 16:05:11.991622,0 days 00:00:00.011395,1.210100,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2026-06-28 16:05:26.572449,2026-06-28 16:05:26.585949,0 days 00:00:00.013500,0.264417,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.763501,2026-06-28 16:05:26.586479,2026-06-28 16:05:26.605684,0 days 00:00:00.019205,0.195117,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2026-06-28 16:05:26.606496,2026-06-28 16:05:26.620422,0 days 00:00:00.013926,0.113085,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-06-28 16:05:26.621729,2026-06-28 16:05:26.635431,0 days 00:00:00.013702,0.167355,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 71
RandomForest        20
GradientBoosting     9
Name: count, dtype: int64

In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.745500
RandomForest        0.770764
SVM                 0.776458
Name: value, dtype: float64

In [30]:
# 1. Optimization History
plot_optimization_history(study).show()

In [31]:
# 3. Slice Plot
plot_slice(study).show()

In [32]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [36]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-06-28 16:06:28,020] A new study created in memory with name: no-name-fe4bdbab-8a35-4dcc-9f65-300dce032463


[0]	train-mlogloss:1.00512	eval-mlogloss:1.00390
[1]	train-mlogloss:0.90822	eval-mlogloss:0.90494
[2]	train-mlogloss:0.86619	eval-mlogloss:0.86507
[3]	train-mlogloss:0.83147	eval-mlogloss:0.82976
[4]	train-mlogloss:0.77274	eval-mlogloss:0.76997
[5]	train-mlogloss:0.74567	eval-mlogloss:0.74260
[6]	train-mlogloss:0.68363	eval-mlogloss:0.67368
[7]	train-mlogloss:0.65368	eval-mlogloss:0.64235
[8]	train-mlogloss:0.61818	eval-mlogloss:0.60438
[9]	train-mlogloss:0.58279	eval-mlogloss:0.56784
[10]	train-mlogloss:0.57415	eval-mlogloss:0.56067
[11]	train-mlogloss:0.54151	eval-mlogloss:0.52385
[12]	train-mlogloss:0.52444	eval-mlogloss:0.50608
[13]	train-mlogloss:0.49763	eval-mlogloss:0.47731
[14]	train-mlogloss:0.48360	eval-mlogloss:0.46035
[15]	train-mlogloss:0.46766	eval-mlogloss:0.44324
[16]	train-mlogloss:0.45116	eval-mlogloss:0.42450
[17]	train-mlogloss:0.44364	eval-mlogloss:0.41524
[18]	train-mlogloss:0.43054	eval-mlogloss:0.40042
[19]	train-mlogloss:0.41933	eval-mlogloss:0.38867
[20]	train

d:\CODING\MACHINE LEARNING\.venv\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  optuna_warn(f"{msg} Use `optuna_integration.xgboost` instead.", FutureWarning)


[112]	train-mlogloss:0.27004	eval-mlogloss:0.20620
[113]	train-mlogloss:0.26990	eval-mlogloss:0.20613
[114]	train-mlogloss:0.26987	eval-mlogloss:0.20621
[115]	train-mlogloss:0.26967	eval-mlogloss:0.20605
[116]	train-mlogloss:0.26965	eval-mlogloss:0.20599
[117]	train-mlogloss:0.26591	eval-mlogloss:0.20125
[118]	train-mlogloss:0.26596	eval-mlogloss:0.20133
[119]	train-mlogloss:0.26567	eval-mlogloss:0.20133
[120]	train-mlogloss:0.26573	eval-mlogloss:0.20147
[121]	train-mlogloss:0.26575	eval-mlogloss:0.20173
[122]	train-mlogloss:0.26573	eval-mlogloss:0.20166
[123]	train-mlogloss:0.26569	eval-mlogloss:0.20126
[124]	train-mlogloss:0.26568	eval-mlogloss:0.20110
[125]	train-mlogloss:0.26563	eval-mlogloss:0.20086
[126]	train-mlogloss:0.26565	eval-mlogloss:0.20092
[127]	train-mlogloss:0.26563	eval-mlogloss:0.20099
[128]	train-mlogloss:0.26563	eval-mlogloss:0.20094
[129]	train-mlogloss:0.26562	eval-mlogloss:0.20100
[130]	train-mlogloss:0.26547	eval-mlogloss:0.20078
[131]	train-mlogloss:0.26547	ev

[I 2026-06-28 16:06:28,469] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.06003346775403882, 'alpha': 4.744688065878044e-06, 'eta': 0.08908659228768515, 'gamma': 2.2743632008920925e-05, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.7322566045414369, 'colsample_bytree': 0.4361896547662477}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.06952	eval-mlogloss:1.07015
[1]	train-mlogloss:1.04061	eval-mlogloss:1.04073
[2]	train-mlogloss:1.01229	eval-mlogloss:1.01019
[3]	train-mlogloss:0.98574	eval-mlogloss:0.98162
[4]	train-mlogloss:0.95966	eval-mlogloss:0.95406
[5]	train-mlogloss:0.93448	eval-mlogloss:0.92820
[6]	train-mlogloss:0.91124	eval-mlogloss:0.90233
[7]	train-mlogloss:0.88812	eval-mlogloss:0.87776


[I 2026-06-28 16:06:31,551] Trial 48 pruned. Trial was pruned at iteration 8.


[0]	train-mlogloss:1.05610	eval-mlogloss:1.05258
[1]	train-mlogloss:1.01812	eval-mlogloss:1.01557


[I 2026-06-28 16:06:31,568] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 0.06003346775403882, 'alpha': 4.744688065878044e-06, 'eta': 0.08908659228768515, 'gamma': 2.2743632008920925e-05, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.7322566045414369, 'colsample_bytree': 0.4361896547662477}
Best accuracy: 1.0


In [37]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()